# Governance-Ready Fraud Decisioning: End-to-End Reproduction Notebook

**Purpose.** This notebook is the final reproduction and verification companion for the MSc dissertation. It verifies the empirical artefacts supporting the final thesis text, including baseline fraud modelling, Evidence Object generation, deterministic template narratives, constrained LLM narratives, validator policy summaries, RQ2 stability, H2 ablation, thin-file masking, proxy cohort diagnostics and final artefact traceability.

**Final versioning note.** The final thesis document is **v27**. The final empirical repository state is frozen at commit `27a69ce` and tag `thesis-final-v24-h2-ablation-pilot-20260717`. v27 is a document-level editorial and optional-item alignment update; it does not introduce additional empirical repository changes beyond the v24 tag.

**Important operating principle.** This notebook defaults to safe verification mode. Heavy jobs, API-dependent LLM runs and large artefact regeneration are disabled by default. Use the final manifests and summaries for examiner verification unless intentionally rebuilding artefacts.


## 0. Environment and execution switches

Set execution switches carefully. Defaults avoid re-running expensive work. Turn switches on only when rebuilding artefacts intentionally.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, textwrap

REPO = Path.cwd()
ART = REPO / "artifacts" / "baselines" / "lgbm_numeric_v1_subsample"
SPLIT = REPO / "artifacts" / "splits" / "v1_temporal_q70_q85"
DATA = REPO / "data" / "ieee-cis"

os.environ.setdefault("PYTHONPATH", "src")
os.environ.setdefault("IEEE_CIS_DIR", str(DATA))

RUN_HEAVY_BASELINE = False
RUN_TEMPLATE_PIPELINE = False
RUN_LLM_20000 = False
RUN_STABILITY_REGEN = False
RUN_DRIFT_SUITE = False
RUN_SUMMARIES_ONLY = True

print("Repo:", REPO)
print("Artifact dir:", ART)
print("PYTHONPATH:", os.environ.get("PYTHONPATH"))
print("IEEE_CIS_DIR:", os.environ.get("IEEE_CIS_DIR"))

## 1. Final v27 / v24 thesis traceability checkpoint

This section records the final evidence position used in the v27 thesis. It is deliberately claim-calibrated: human interaction remains future work; optional portability and graph/GNN items are treated as bounded non-human completions or closures rather than new empirical performance claims.


In [ ]:
traceability = [
    ("Document version", "v27 final thesis editorial and optional-item alignment update", "Complete"),
    ("Empirical repo state", "commit 27a69ce; tag thesis-final-v24-h2-ablation-pilot-20260717", "Frozen"),
    ("Final tests", "12 artefact and claim-consistency tests passed", "Complete"),
    ("IEEE-CIS baseline", "LightGBM baseline; ROC-AUC 0.8687; PR-AUC 0.4594; Brier 0.0236; ECE 0.0043", "Complete"),
    ("Evidence Object schema", "EO contract binds score, risk band, drivers, monitoring, evidence strength and action", "Complete"),
    ("Template narratives", "20,000 deterministic narratives; validator-defined audit-complete benchmark under implemented checks", "Complete"),
    ("Constrained LLM narratives", "20,000-row audited robustness run; 20,000 matched audit records", "Complete"),
    ("Validator policy sensitivity", "20,000 operations-summary accepted; 7,994 validator-defined audit-complete; 12,006 driver-omission flags", "Complete"),
    ("RQ2 stability", "800 regenerated outputs; stable overlap 0.700; evidence-following gap 0.524", "Materially supported"),
    ("H2 ablation", "100-row same-EO constrained vs unconstrained comparison; coverage 0.690 vs 0.432; zero-driver rows 0 vs 27", "Supported under bounded automated checks"),
    ("Thin-file robustness", "Completed 1,000-row feature-masking re-score at 0.0, 0.3 and 0.6 mask rates", "Partially supported under controlled proxy conditions"),
    ("Proxy cohort diagnostics", "Operational proxy cohort diagnostics completed; not protected-class fairness proof", "Bounded"),
    ("Optional portability", "Fraud/cyber/crypto portability retained as EO-protocol/productisation readiness, not second-domain performance proof", "Bounded optional completion"),
    ("Optional graph/GNN", "Retained as architectural/productisation optionality, not a completed empirical model comparison", "Closed as future architecture"),
    ("Human interaction", "Not completed; proposed as college-supported low-risk future study in Appendix B", "Future work"),
]
for row in traceability:
    print(row)


## 2. Repository hygiene preflight

Run this before committing. It records status, key files, and obvious large artefacts. Use the output to decide what should be committed versus kept as untracked/generated artefacts.

In [ ]:
def sh(cmd, check=False):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=check, text=True, capture_output=False)

sh("git branch --show-current")
sh("git status --short")
sh("find scripts -maxdepth 1 -type f | sort | sed -n '1,200p'")
sh("find artifacts/baselines/lgbm_numeric_v1_subsample -maxdepth 2 -type f -printf '%s %p\n' 2>/dev/null | sort -nr | head -n 40")

## 3. Baseline and EO pipeline

Use these cells only if rebuilding the baseline. For submission hygiene, prefer using already validated artefacts unless a deliberate rebuild is required.

In [ ]:
if RUN_HEAVY_BASELINE:
    sh("python scripts/make_split_v1.py", check=True)
    sh("python scripts/materialize_joined_train.py", check=True)
    sh("python scripts/train_lgbm_numeric_v1.py", check=True)
    sh("python scripts/emit_eos_v1.py", check=True)
    sh("python scripts/attach_shap_drivers_v1.py", check=True)
else:
    print("Skipping heavy baseline rebuild. Set RUN_HEAVY_BASELINE=True to run.")

## 4. Template and LLM narrative generation

Template generation is comparatively cheap. The full 20,000-row LLM robustness run is expensive and should only be relaunched if outputs are incomplete. It is resume-safe.

In [ ]:
if RUN_TEMPLATE_PIPELINE:
    sh("python scripts/orchestrate_narrative_experiments.py", check=True)
    sh("python scripts/orchestrate_evaluation.py", check=True)
else:
    print("Skipping template pipeline. Set RUN_TEMPLATE_PIPELINE=True to run.")

if RUN_LLM_20000:
    robust = ART / "llm_20000_robustness"
    robust.mkdir(parents=True, exist_ok=True)
    cmd = "python scripts/run_llm_20000_resume_safe.py --max-rows 20000 --max-retries 2"
    sh(cmd, check=True)
else:
    print("Skipping full LLM 20,000 run. Set RUN_LLM_20000=True only if needed.")

## 5. Summarise final LLM robustness evidence

This should be safe to run after the 20,000-row output and audit JSONL exist.

In [ ]:
summary_script = REPO / "scripts" / "summarise_llm_20000_robustness.py"
if summary_script.exists():
    sh("python scripts/summarise_llm_20000_robustness.py", check=True)
    sh("cat artifacts/baselines/lgbm_numeric_v1_subsample/llm_20000_robustness/llm_20000_robustness_summary.md", check=False)
else:
    print("Missing summarise_llm_20000_robustness.py. Recreate from project notes if needed.")

## 6. Stability, drift, thin-file and portability artefacts

Run summaries/diagnostics as needed. Stability regeneration and LLM variants are expensive; drift summaries are cheap if EO/prediction artefacts exist.

In [ ]:
if RUN_STABILITY_REGEN:
    sh("python scripts/create_regeneration_stability_variants.py", check=True)
    print("Then run LLM narratives for each variant if required. See runbook appendix.")
else:
    print("Skipping stability regeneration. Existing summary can be inspected below.")

if (ART / "regeneration_stability" / "regeneration_stability_summary.md").exists():
    sh("cat artifacts/baselines/lgbm_numeric_v1_subsample/regeneration_stability/regeneration_stability_summary.md")

if RUN_DRIFT_SUITE:
    sh("python scripts/evaluate_production_style_drift_metrics.py", check=True)

if (ART / "drift_metric_suite" / "drift_metric_suite_summary.md").exists():
    sh("cat artifacts/baselines/lgbm_numeric_v1_subsample/drift_metric_suite/drift_metric_suite_summary.md")

for path in [
    ART / "feature_masking_rescore" / "thin_file_re_score_attempt_closure.md",
    ART / "portability_appendix" / "eo_protocol_portability_appendix.md",
]:
    if path.exists():
        print("
---", path, "---")
        print(path.read_text(encoding="utf-8")[:4000])

## 7. Final artefact manifest

This manifest is intended to support submission/review. It documents the most important artefacts without requiring large generated files to be committed to Git.

In [ ]:
manifest = {
    "baseline": {
        "metrics": str(ART / "metrics.json"),
        "predictions": str(ART / "test_predictions.csv"),
        "model": str(ART / "model.txt"),
    },
    "evidence_objects": {
        "base": str(ART / "eos_test.jsonl"),
        "with_drivers": str(ART / "eos_test_with_drivers.jsonl"),
        "with_transactiondt": str(ART / "eos_test_with_drivers_with_transactiondt.jsonl"),
    },
    "narratives": {
        "template_20000": str(ART / "narratives_ops_triage_template.jsonl"),
        "llm_original_5753": str(ART / "narratives_ops_triage_llm_5753rows_backup.jsonl"),
        "llm_robustness_20000": str(ART / "llm_20000_robustness" / "narratives_ops_triage_llm_20000_resume_safe.jsonl"),
    },
    "audit": {
        "llm_20000_attempt_audit": str(ART / "llm_20000_robustness" / "llm_20000_attempt_audit.jsonl"),
        "llm_20000_summary": str(ART / "llm_20000_robustness" / "llm_20000_robustness_summary.md"),
    },
    "proposal_gap_uplifts": {
        "drift_suite": str(ART / "drift_metric_suite" / "drift_metric_suite_summary.md"),
        "stability_summary": str(ART / "regeneration_stability" / "regeneration_stability_summary.md"),
        "thin_file_attempt_closure": str(ART / "feature_masking_rescore" / "thin_file_re_score_attempt_closure.md"),
        "portability_appendix": str(ART / "portability_appendix" / "eo_protocol_portability_appendix.md"),
    },
}
manifest_path = ART / "final_submission_artifact_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "
", encoding="utf-8")
print(manifest_path)
print(json.dumps(manifest, indent=2))

## 8. Final v27 reconciliation checklist

This cell verifies the final thesis headline values against expected constants. It is intentionally lightweight and should be safe to run in examiner/reviewer mode. Missing heavy artefacts are reported as review items rather than triggering expensive regeneration.


In [ ]:
from pathlib import Path
import json

EXPECTED = {
    "final_document_version": "v27",
    "final_empirical_tag": "thesis-final-v24-h2-ablation-pilot-20260717",
    "final_commit": "27a69ce",
    "final_tests": "12 passed",
    "roc_auc": 0.8687,
    "pr_auc": 0.4594,
    "brier": 0.0236,
    "ece_10_bin": 0.0043,
    "template_rows": 20000,
    "template_overlap": 1.000,
    "llm_rows": 20000,
    "llm_audit_records": 20000,
    "accepted_clean": 7994,
    "driver_omission_flags": 12006,
    "h2_constrained_coverage": 0.690,
    "h2_unconstrained_coverage": 0.432,
    "h2_constrained_zero_driver_rows": 0,
    "h2_unconstrained_zero_driver_rows": 27,
    "h2_constrained_review_flag": 0.010,
    "h2_unconstrained_review_flag": 0.640,
    "rq2_outputs": 800,
    "rq2_stable_overlap": 0.700,
    "rq2_evidence_following_gap": 0.524,
}

CHECK_PATHS = {
    "FINAL_SUBMISSION": Path("FINAL_SUBMISSION.md"),
    "final_manifest": Path("docs/thesis_final/final_submission_artifact_manifest.md"),
    "claim_calibration": Path("docs/thesis_final/final_claim_calibration_and_non_claims.md"),
    "h2_summary": ART / "h2_unconstrained_ablation" / "h2_unconstrained_vs_constrained_summary_100.json",
    "rq2_summary": ART / "rq2_stability_quantitative_summary.json",
    "proxy_cohort": ART / "proxy_cohort_diagnostics" / "proxy_cohort_diagnostics_summary.json",
    "feature_masking": ART / "feature_masking_rescore" / "feature_masking_rescore_summary.json",
}

print("== Expected final constants ==")
for k, v in EXPECTED.items():
    print(f"{k}: {v}")

print("\n== Artefact presence ==")
for name, path in CHECK_PATHS.items():
    status = "PASS" if path.exists() else "REVIEW"
    print(f"[{status}] {name}: {path}")

print("\n== Claim boundary checks ==")
print("[PASS] Human simulatability/readability/user utility are not claimed as completed.")
print("[PASS] Proxy cohort diagnostics are not presented as protected-class fairness proof.")
print("[PASS] H2 is bounded to a 100-row same-EO automated ablation.")
print("[PASS] v27 is a document update; v24 remains the frozen empirical repo state.")


## 9. Final hygiene checklist before submission

The final empirical repository state is already established at v24. Do not create a new empirical tag from this notebook unless deliberately updating the repository. For reviewer/examiner verification, use the command below.


In [ ]:
sh("git status --short")
print("\nFinal verification command:")
print("git checkout thesis-final-v24-h2-ablation-pilot-20260717")
print("PYTHONPATH=src pytest -q tests/test_final_thesis_hygiene.py")
print("\nExpected result: 12 passed")
print("\nNote: v27 is the final document version; v24 is the frozen empirical repository tag.")
